# 03 · Launch TorchDistributor — Multi Node

Multi-node 토폴로지(M×N). 학습 함수는 [`torch_distributor_trainer.py`](./torch_distributor_trainer.py)의 `train_fn`이며 1×1·1×N과 동일합니다. 차이는 `local_mode=False`와 `num_processes = M*N`.

> `local_mode=False`는 Spark worker 노드들에 child 프로세스를 분산 spawn합니다. driver는 학습에 참여하지 않으므로 driver-side `log_system_metrics`만 켜면 idle metric만 잡힙니다 — `train_fn`은 rank-0 worker에서 `mlflow.start_run(run_id=..., log_system_metrics=True)`로 attach해 worker GPU 메트릭을 함께 기록합니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML. Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**, Single node 토글 OFF. Serverless GPU는 multi-node 미지원이라 사용 불가.

함정 모음: [`00-foundations/common-pitfalls.md`](../00-foundations/common-pitfalls.md). worker 노드에서 sibling `.py` import를 위해 `SCRIPT_DIR`을 인자로 넘기고 child 함수가 `sys.path`에 삽입합니다 ([`§2`](../00-foundations/common-pitfalls.md)).

In [ ]:
%run ./00-setup

## 학습 함수 import + SCRIPT_DIR

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

from torch_distributor_trainer import train_fn

## M×N GPU

`NUM_NODES`, `NUM_GPUS_PER_NODE`를 클러스터 구성에 맞춥니다. 1×1·1×N과 같은 `CONFIG`·같은 `DATA_DIR` 사용.

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2          # M (worker 노드 수)
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-MxN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="MxN",
    script_dir=SCRIPT_DIR,
)